# Regulatory Knowledge Graph (GraphRAG) — Dev Log

## Objetivo e papel no pipeline

`core/regulatory_knowledge_graph` evolui o `core/regulatory_rag` (V1, busca
por similaridade pura) para um **grafo de conhecimento navegável** — consolida
duas linhas do ROADMAP original ("GraphRAG" e "Regulatory Knowledge Graph"),
que são, na prática, a mesma capacidade.

**Nenhuma aresta é inventada.** O grafo é construído extraindo, por regex,
menções textuais reais a outros artigos dentro do corpo de cada arquivo do
corpus (`core/regulatory_rag/corpus/*.txt`) — ex.: `art_38_ripd.txt` menciona
literalmente "Art. 11º" e "Art. 20º" no texto, e é isso que vira as arestas
`art_38 -> art_11` e `art_38 -> art_20`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.regulatory_knowledge_graph.graph import build_graph, find_related, shortest_path, to_result

graph = build_graph()
result = to_result(graph)
print(f"Grafo: {result.node_count} nós, {result.edge_count} arestas")
print()
print("Arestas reais extraídas do corpus (amostra):")
for e in result.edges[:6]:
    print(f"  {e.source} --[{e.relation}]--> {e.target}")
print()
related = find_related(graph, "38", max_hops=2)
print("Artigos relacionados ao Art. 38º (RIPD), até 2 saltos:")
for r in related:
    print(f"  distância={r['distance']} | {r['id']} | Art. {r['article']} | {r['tema']}")
print()
path = shortest_path(graph, "38", "9")
print("Caminho mais curto Art. 38º -> Art. 9º:", path)

Grafo: 12 nós, 10 arestas

Arestas reais extraídas do corpus (amostra):
  art_11 --[menciona]--> art_7
  art_11 --[menciona]--> art_5
  art_12 --[menciona]--> art_5
  art_18 --[menciona]--> art_20
  art_20 --[menciona]--> art_9
  art_38 --[menciona]--> art_20

Artigos relacionados ao Art. 38º (RIPD), até 2 saltos:
  distância=1 | art_20 | Art. 20º | Decisões automatizadas sobre a pessoa (direito de revisão)
  distância=1 | art_11 | Art. 11º | Bases legais para tratamento de dado pessoal sensível
  distância=2 | art_9 | Art. 9º | Direito de acesso a informações sobre o tratamento (transparência)
  distância=2 | art_18 | Art. 18º | Direitos do titular dos dados
  distância=2 | art_5 | Art. 5º | Definições (dado pessoal, dado sensível, anonimização, tratamento)
  distância=2 | art_7 | Art. 7º | Bases legais gerais para tratamento de dados pessoais

Caminho mais curto Art. 38º -> Art. 9º: ['art_38', 'art_20', 'art_9']


O caminho `Art. 38º -> Art. 20º -> Art. 9º` é real: o corpus do Art. 38º
(RIPD) menciona textualmente o Art. 20º (decisões automatizadas), que por sua
vez menciona o Art. 9º (transparência) — uma cadeia de dependência
regulatória que a busca por similaridade pura do `regulatory_rag` não
consegue expressar diretamente (ela rankeia por proximidade semântica, não
por relação estrutural entre artigos).

## Rodando a suíte de testes

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/regulatory_knowledge_graph/tests -v
```

12 testes contra o corpus **real**: 12 nós esperados; arestas reais
conhecidas; sem self-loops; atributos corretos; `to_result` consistente;
`find_related` em 1 e 2 saltos; artigo desconhecido; caminhos direto,
multi-hop e inexistente; grafo dirigido.

## Handoff Summary

- **Status:** ✅ done — 12/12 testes passando.
- **TODO onda futura:** GraphRAG híbrido de verdade (combinar este grafo com
  `regulatory_rag.query()` — recuperação por similaridade + expansão por
  grafo), hoje são dois módulos independentes.